In [1]:
import openai
from pycparser import preprocess_file
from qdrant_client import QdrantClient

In [1]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding

### retrieval function

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [5]:
def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-item-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

In [2]:
retrieve_context = retrieve_data("what kind of earphones can I get?", qdrant_client, k=10)

NameError: name 'retrieve_data' is not defined

In [7]:
retrieve_context


{'retrieved_context_ids': ['B0CBMPG524',
  'B0B9FTVL58',
  'B0C142QS8X',
  'B0C6K1GQCF',
  'B0BNHVLF7G',
  'B09WCFC5D9',
  'B0B14HTZ59',
  'B0B67ZFRPC',
  'B0B7495RL6',
  'B0BS15TRJ3'],
 'retrieved_context': ['Open Ear Headphones, Bluetooth 5.3 Earbuds with 60H Playtime IPX7 Waterproof Wireless Earbuds Immersive Premium Sound True Wireless Open Ear Earbuds with Earhooks for Running, Walking and Workouts 【Open-ear Design Headphones】Feature with a new generation of true open-ear wireless earbuds design, the headphones can rest gently and firmly fit your ears without entering your ear canal, which will reduce stress and hearing loss after extended wear. There is no pinching of the auricle, no blockage of the ear canal, and no pain or damage to hearing. 【Powerful Stereo Sound】Equipped with 16.2 millimeters vibrating diaphragm speaker driver, bluetooth headphones providing pure balanced audio and clarity output for all music genres with soft audio and immerse yourself in the wonderful world

### Format retrieved context function

In [8]:
def process_context(context):
    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"],
                                 context["retrieved_context_ratings"]):
        formatted_context += f"-ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [9]:
preprocessed_context = process_context(retrieve_context)

In [11]:
print(preprocessed_context)

-ID: B0CBMPG524, rating: 4.4, description: Open Ear Headphones, Bluetooth 5.3 Earbuds with 60H Playtime IPX7 Waterproof Wireless Earbuds Immersive Premium Sound True Wireless Open Ear Earbuds with Earhooks for Running, Walking and Workouts 【Open-ear Design Headphones】Feature with a new generation of true open-ear wireless earbuds design, the headphones can rest gently and firmly fit your ears without entering your ear canal, which will reduce stress and hearing loss after extended wear. There is no pinching of the auricle, no blockage of the ear canal, and no pain or damage to hearing. 【Powerful Stereo Sound】Equipped with 16.2 millimeters vibrating diaphragm speaker driver, bluetooth headphones providing pure balanced audio and clarity output for all music genres with soft audio and immerse yourself in the wonderful world of music. 【Hear Your Surroundings】Open earbuds that rest on your ears without covering them, you can hear your music and your surroundings at the same time. Whether y

### create promt function

In [12]:
def build_prompt(preprocessed_context, question):
    prompt = f"""
        You are a shopping assistant that can answer question about products in stock
        You will be given a question and a list of context.

        Instructions:
        - You need to answer question based on the provided context only.
        - Never use word context and refer to it as the available products.

        Context:
        {preprocessed_context}

        Question:
        {question}
        """

    return prompt


In [13]:
prompt = build_prompt(preprocessed_context, "what kind of earphones can I get?")

In [15]:
print(prompt)


        You are a shopping assistant that can answer question about products in stock
        You will be given a question and a list of context.

        Instructions:
        - You need to answer question based on the provided context only.
        - Never use word context and refer to it as the available products.

        Context:
        -ID: B0CBMPG524, rating: 4.4, description: Open Ear Headphones, Bluetooth 5.3 Earbuds with 60H Playtime IPX7 Waterproof Wireless Earbuds Immersive Premium Sound True Wireless Open Ear Earbuds with Earhooks for Running, Walking and Workouts 【Open-ear Design Headphones】Feature with a new generation of true open-ear wireless earbuds design, the headphones can rest gently and firmly fit your ears without entering your ear canal, which will reduce stress and hearing loss after extended wear. There is no pinching of the auricle, no blockage of the ear canal, and no pain or damage to hearing. 【Powerful Stereo Sound】Equipped with 16.2 millimeters vibrati

### GENERATE answer function

In [7]:
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="minimal"
    )

    return response.choices[0].message.content

In [17]:
print(generate_answer(prompt))

Here are earphone options currently available in the available products:

- Open Ear Headphones (B0CBMPG524): Open-ear wireless earbuds with Bluetooth 5.3, 60 hours with case, IPX7 waterproof, includes earhooks for secure fit.
- Wireless Earbuds with LED display (B0B9FTVL58): Bluetooth 5.3 in-ear headphones, 37 hours playback, IPX7 waterproof, smart touch controls, charging case.
- TELSOR Wireless Earbuds (B0C6K1GQCF): Bluetooth 5.1, 30 hours total with case, IPX7 waterproof, smart touch controls, noise-cancelling mic.
- Siniffo X15 Bone Conduction Headphones (B0BNHVLF7G): Open-ear bone conduction headphones, IP56 waterproof, bone-conduction sound.

If you’re looking for traditional in-ear earbuds, the first two options are typical in-ear styles, while the Siniffo option is bone-conduction open-ear. Let me know your preferred style, budget, and key features (waterproof, battery life, mic, etc.), and I can help narrow it further.


### Combined RAG Pipeline

In [20]:
def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieve_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieve_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer

In [22]:
print(rag_pipeline("What kind of earphones can I get with ratings above 4.4?"))

Based on the available products, the earphones with a rating above 4.4 are:

- B0CBMPG524: Open Ear Headphones rated 4.4 (meets the threshold if strictly “above 4.4” would exclude 4.4; but since asked “above 4.4,” this one is not included). If you interpret as 4.4 or higher, this qualifies.
- B0C142QS8X: TUNEAKE Kids Headphones rated 4.5 (above 4.4).

If you strictly need ratings greater than 4.4, the only item is:
- TUNEAKE Kids Headphones (B0C142QS8X) with a 4.5 rating.

If you meant 4.4 or higher, add:
- Open Ear Headphones (B0CBMPG524) with a 4.4 rating
- TUNEAKE Kids Headphones (B0C142QS8X) with 4.5 rating

Would you like details on features for the qualifying option?


In [5]:
import openai
print("module api_key:", getattr(openai, "api_key", None))

module api_key: None
